In [1]:
print("[BOOT] CrowdDensity FULL DATA v4 execution started", flush=True)


[BOOT] CrowdDensity FULL DATA v4 execution started


In [2]:
import time
print("[BOOT] heartbeat A", flush=True)
time.sleep(1)
print("[BOOT] heartbeat B", flush=True)


[BOOT] heartbeat A
[BOOT] heartbeat B


In [3]:
import os, sys, time, json, random, shutil, warnings, subprocess, importlib.util
from pathlib import Path

warnings.filterwarnings("ignore")

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

log("[BOOT] Full-data pipeline imports/config started")

SEED = 42
random.seed(SEED)

DATA_ROOT = Path("/kaggle/input/datasets/wafabila/crowdhuman-datset")

WORKDIR = Path("/kaggle/working/crowddensity_full_v4")
YOLO_ROOT = WORKDIR / "crowdhuman_yolo"
REPORT_DIR = WORKDIR / "reports"
MODEL_DIR = WORKDIR / "models"

# Final package is kept separate so the temporary linked dataset
# is NOT accidentally included in the ZIP.
OUTPUT_DIR = Path("/kaggle/working/crowddensity_full_v4_output")

for p in [WORKDIR, YOLO_ROOT, REPORT_DIR, MODEL_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

EPOCHS = 15
IMG_SIZE = 640
BATCH = 16
WORKERS = 2

log(f"[BOOT] DATA_ROOT={DATA_ROOT}")
log("[BOOT] MODE=FULL ANNOTATED DATA")
log(f"[BOOT] epochs={EPOCHS}, imgsz={IMG_SIZE}, batch={BATCH}")

[12:28:09] [BOOT] Full-data pipeline imports/config started
[12:28:09] [BOOT] DATA_ROOT=/kaggle/input/datasets/wafabila/crowdhuman-datset
[12:28:09] [BOOT] MODE=FULL ANNOTATED DATA
[12:28:09] [BOOT] epochs=15, imgsz=640, batch=16


In [4]:
log("[1/10] Environment check started")

import torch

log(f"[1/10] Torch={torch.__version__}")
log(f"[1/10] CUDA={torch.cuda.is_available()}")
log(f"[1/10] GPU count={torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        log(f"[1/10] GPU {i}: {torch.cuda.get_device_name(i)}")

assert DATA_ROOT.exists(), f"Dataset root not found: {DATA_ROOT}"

for p in sorted(DATA_ROOT.iterdir()):
    log(f"[1/10] input: {p.name}")

log("[1/10] Environment check complete")

[12:28:09] [1/10] Environment check started
[12:28:17] [1/10] Torch=2.11.0+cu128
[12:28:17] [1/10] CUDA=True
[12:28:17] [1/10] GPU count=2
[12:28:17] [1/10] GPU 0: Tesla T4
[12:28:17] [1/10] GPU 1: Tesla T4
[12:28:17] [1/10] input: CrowdHuman_test
[12:28:17] [1/10] input: CrowdHuman_train01
[12:28:17] [1/10] input: CrowdHuman_train02
[12:28:17] [1/10] input: CrowdHuman_train03
[12:28:17] [1/10] input: CrowdHuman_val
[12:28:17] [1/10] input: README.md
[12:28:17] [1/10] input: annotation_train.odgt
[12:28:17] [1/10] input: annotation_val.odgt
[12:28:17] [1/10] Environment check complete


In [5]:
log("[2/10] Dependency check started")

def ensure(pkg, mod):
    if importlib.util.find_spec(mod) is None:
        log(f"[2/10] Installing {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    else:
        log(f"[2/10] {mod} already installed")

ensure("ultralytics>=8.3.0", "ultralytics")

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = True

log("[2/10] Imports complete")

[12:28:17] [2/10] Dependency check started
[12:28:17] [2/10] Installing ultralytics>=8.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 kB 3.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[12:28:25] [2/10] Imports complete


In [6]:
log("[3/10] Locating CrowdHuman images")

TRAIN_ANN = DATA_ROOT / "annotation_train.odgt"
VAL_ANN = DATA_ROOT / "annotation_val.odgt"

TRAIN_DIRS = [
    DATA_ROOT / "CrowdHuman_train01",
    DATA_ROOT / "CrowdHuman_train02",
    DATA_ROOT / "CrowdHuman_train03",
]
VAL_DIRS = [DATA_ROOT / "CrowdHuman_val"]

assert TRAIN_ANN.exists()
assert VAL_ANN.exists()

def find_images(root):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

train_images = []
for d in TRAIN_DIRS:
    if d.exists():
        found = find_images(d)
        train_images.extend(found)
        log(f"[3/10] {d.name}: {len(found)} images")

val_images = []
for d in VAL_DIRS:
    if d.exists():
        found = find_images(d)
        val_images.extend(found)
        log(f"[3/10] {d.name}: {len(found)} images")

log(f"[3/10] Total train={len(train_images)}")
log(f"[3/10] Total val={len(val_images)}")

assert train_images and val_images

log("[3/10] Image discovery complete")

[12:28:25] [3/10] Locating CrowdHuman images
[12:28:33] [3/10] CrowdHuman_train01: 5000 images
[12:28:41] [3/10] CrowdHuman_train02: 5000 images
[12:28:50] [3/10] CrowdHuman_train03: 5000 images
[12:28:56] [3/10] CrowdHuman_val: 4370 images
[12:28:56] [3/10] Total train=15000
[12:28:56] [3/10] Total val=4370
[12:28:56] [3/10] Image discovery complete


In [7]:
log("[4/10] Parsing ODGT")

def read_odgt(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if line:
                records.append(json.loads(line))
            if i % 5000 == 0:
                log(f"[4/10] {path.name}: {i} lines parsed")
    return records

train_records = read_odgt(TRAIN_ANN)
val_records = read_odgt(VAL_ANN)

train_lookup = {p.stem: p for p in train_images}
val_lookup = {p.stem: p for p in val_images}

def valid_person_boxes(record):
    out = []
    for gt in record.get("gtboxes", []):
        if int(gt.get("extra", {}).get("ignore", 0)) == 1:
            continue
        box = gt.get("fbox")
        if not box or len(box) != 4:
            continue
        x, y, w, h = map(float, box)
        if w > 1 and h > 1:
            out.append((x, y, w, h))
    return out

log(f"[4/10] train records={len(train_records)}")
log(f"[4/10] val records={len(val_records)}")
log("[4/10] Parsing complete")

[12:28:56] [4/10] Parsing ODGT
[12:28:58] [4/10] annotation_train.odgt: 5000 lines parsed
[12:28:59] [4/10] annotation_train.odgt: 10000 lines parsed
[12:29:00] [4/10] annotation_train.odgt: 15000 lines parsed
[12:29:02] [4/10] train records=15000
[12:29:02] [4/10] val records=4370
[12:29:02] [4/10] Parsing complete


In [8]:
log("[5/10] Using ALL annotated CrowdHuman data + converting labels")

train_available = [r for r in train_records if r["ID"] in train_lookup]
val_available = [r for r in val_records if r["ID"] in val_lookup]

# FULL DATA: no subsampling.
train_selected = train_available
val_selected = val_available

log(f"[5/10] FULL train selected={len(train_selected)}")
log(f"[5/10] FULL val selected={len(val_selected)}")

assert len(train_selected) == len(train_records), (
    f"Expected all {len(train_records)} train records, "
    f"but only matched {len(train_selected)} images."
)
assert len(val_selected) == len(val_records), (
    f"Expected all {len(val_records)} val records, "
    f"but only matched {len(val_selected)} images."
)

for split in ["train", "val"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

def clip(v, lo, hi):
    return max(lo, min(v, hi))

def link_or_copy(src, dst):
    if dst.exists() or dst.is_symlink():
        return "exists"
    try:
        os.symlink(src, dst)
        return "symlink"
    except Exception:
        shutil.copy2(src, dst)
        return "copy"

def convert(records, lookup, split):
    rows = []
    total = len(records)
    linked = 0
    copied = 0
    skipped = 0

    for i, rec in enumerate(records, 1):
        src = lookup.get(rec["ID"])
        if src is None:
            skipped += 1
            continue

        try:
            with Image.open(src) as im:
                W, H = im.size
        except Exception:
            skipped += 1
            continue

        dst = YOLO_ROOT / "images" / split / src.name
        mode = link_or_copy(src, dst)

        if mode == "symlink":
            linked += 1
        elif mode == "copy":
            copied += 1

        lines = []

        for x, y, w, h in valid_person_boxes(rec):
            x1 = clip(x, 0, W - 1)
            y1 = clip(y, 0, H - 1)
            x2 = clip(x + w, 0, W)
            y2 = clip(y + h, 0, H)

            bw = x2 - x1
            bh = y2 - y1

            if bw <= 1 or bh <= 1:
                continue

            xc = (x1 + x2) / 2 / W
            yc = (y1 + y2) / 2 / H
            nw = bw / W
            nh = bh / H

            lines.append(
                f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}"
            )

        (YOLO_ROOT / "labels" / split / f"{src.stem}.txt").write_text(
            "\n".join(lines),
            encoding="utf-8"
        )

        rows.append({
            "split": split,
            "filename": src.name,
            "persons": len(lines),
            "width": W,
            "height": H,
        })

        if i == 1 or i % 500 == 0 or i == total:
            log(
                f"[5/10] {split}: {i}/{total} | "
                f"linked={linked} copied={copied} skipped={skipped}"
            )

    return pd.DataFrame(rows)

train_manifest = convert(train_selected, train_lookup, "train")
val_manifest = convert(val_selected, val_lookup, "val")

manifest = pd.concat(
    [train_manifest, val_manifest],
    ignore_index=True
)

manifest.to_csv(
    REPORT_DIR / "manifest.csv",
    index=False
)

log(f"[5/10] FINAL train manifest={len(train_manifest)}")
log(f"[5/10] FINAL val manifest={len(val_manifest)}")

assert len(train_manifest) == 15000, (
    f"Expected 15000 train images, got {len(train_manifest)}"
)
assert len(val_manifest) == 4370, (
    f"Expected 4370 val images, got {len(val_manifest)}"
)

log("[5/10] FULL DATA conversion complete")

[12:29:02] [5/10] Using ALL annotated CrowdHuman data + converting labels
[12:29:02] [5/10] FULL train selected=15000
[12:29:02] [5/10] FULL val selected=4370
[12:29:02] [5/10] train: 1/15000 | linked=1 copied=0 skipped=0
[12:29:05] [5/10] train: 500/15000 | linked=500 copied=0 skipped=0
[12:29:08] [5/10] train: 1000/15000 | linked=1000 copied=0 skipped=0
[12:29:11] [5/10] train: 1500/15000 | linked=1500 copied=0 skipped=0
[12:29:14] [5/10] train: 2000/15000 | linked=2000 copied=0 skipped=0
[12:29:16] [5/10] train: 2500/15000 | linked=2500 copied=0 skipped=0
[12:29:19] [5/10] train: 3000/15000 | linked=3000 copied=0 skipped=0
[12:29:22] [5/10] train: 3500/15000 | linked=3500 copied=0 skipped=0
[12:29:24] [5/10] train: 4000/15000 | linked=4000 copied=0 skipped=0
[12:29:27] [5/10] train: 4500/15000 | linked=4500 copied=0 skipped=0
[12:29:30] [5/10] train: 5000/15000 | linked=5000 copied=0 skipped=0
[12:29:33] [5/10] train: 5500/15000 | linked=5500 copied=0 skipped=0
[12:29:35] [5/10] tra

In [9]:
log("[6/10] Creating dataset YAML")

DATA_YAML = YOLO_ROOT / "crowdhuman.yaml"
DATA_YAML.write_text(
    f"""path: {YOLO_ROOT}
train: images/train
val: images/val

names:
  0: person
""",
    encoding="utf-8"
)

print(DATA_YAML.read_text(), flush=True)

assert len(train_manifest) > 0
assert len(val_manifest) > 0

log("[6/10] YAML ready")

[12:30:50] [6/10] Creating dataset YAML
path: /kaggle/working/crowddensity_full_v4/crowdhuman_yolo
train: images/train
val: images/val

names:
  0: person

[12:30:50] [6/10] YAML ready


In [10]:
log("[7/10] Loading YOLO11n")

model = YOLO("yolo11n.pt")

log("[7/10] YOLO11n loaded")
log("[7/10] Training will use GPU 0 for notebook stability")

[12:30:50] [7/10] Loading YOLO11n
[12:30:51] [7/10] YOLO11n loaded
[12:30:51] [7/10] Training will use GPU 0 for notebook stability


In [11]:
log("[8/10] FULL DATA TRAINING START")
log(
    f"[8/10] epochs={EPOCHS}, imgsz={IMG_SIZE}, "
    f"batch={BATCH}, workers={WORKERS}"
)

started = time.time()

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=WORKERS,
    project=str(WORKDIR / "runs"),
    name="crowdhuman_yolo11n_full_v4",
    pretrained=True,
    optimizer="auto",
    patience=5,
    cache=False,
    seed=SEED,
    deterministic=True,
    verbose=True,
    save=True,
    plots=True,
    max_det=400,
)

log(f"[8/10] TRAINING COMPLETE in {(time.time()-started)/3600:.2f} h")

[12:30:51] [8/10] FULL DATA TRAINING START
[12:30:51] [8/10] epochs=15, imgsz=640, batch=16, workers=2
Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/crowddensity_full_v4/crowdhuman_yolo/crowdhuman.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=400, mixup=0.

In [12]:
log("[9/10] Full validation + count demo started")

BEST_MODEL = (
    WORKDIR
    / "runs"
    / "crowdhuman_yolo11n_full_v4"
    / "weights"
    / "best.pt"
)

assert BEST_MODEL.exists(), BEST_MODEL

best = YOLO(str(BEST_MODEL))

m = best.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=WORKERS,
    verbose=True,
    max_det=400,
)

metrics = {
    "precision": float(m.box.mp),
    "recall": float(m.box.mr),
    "mAP50": float(m.box.map50),
    "mAP50_95": float(m.box.map),
}

print(json.dumps(metrics, indent=2), flush=True)

# Counting demo on 50 random annotated validation images.
demo = val_manifest.sample(
    n=min(50, len(val_manifest)),
    random_state=SEED
).reset_index(drop=True)

rows = []

for i, row in demo.iterrows():
    img = YOLO_ROOT / "images" / "val" / row["filename"]

    pred = best.predict(
        source=str(img),
        imgsz=IMG_SIZE,
        conf=0.25,
        iou=0.5,
        max_det=400,
        verbose=False
    )[0]

    pred_count = len(pred.boxes)
    gt_count = int(row["persons"])

    rows.append({
        "filename": row["filename"],
        "ground_truth_count": gt_count,
        "predicted_count": pred_count,
        "absolute_error": abs(pred_count - gt_count),
    })

    if i == 0 or (i + 1) % 10 == 0 or i + 1 == len(demo):
        log(
            f"[9/10] counting demo {i+1}/{len(demo)} | "
            f"GT={gt_count} pred={pred_count}"
        )

count_df = pd.DataFrame(rows)

count_df.to_csv(
    REPORT_DIR / "counting_demo_50.csv",
    index=False
)

count_mae = float(
    count_df["absolute_error"].mean()
)

log(f"[9/10] Demo Count MAE (50 images)={count_mae:.4f}")

# Qualitative inference on official CrowdHuman test images.
# No test GT annotation is attached, so these are NOT used for metrics.
TEST_DIR = DATA_ROOT / "CrowdHuman_test"

test_images = [
    p for p in TEST_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
] if TEST_DIR.exists() else []

test_demo_dir = OUTPUT_DIR / "test_demo"
test_demo_dir.mkdir(parents=True, exist_ok=True)

if test_images:
    rng = random.Random(SEED)
    selected_test = rng.sample(
        test_images,
        k=min(20, len(test_images))
    )

    log(
        f"[9/10] Running qualitative inference on "
        f"{len(selected_test)} official test images"
    )

    for i, img_path in enumerate(selected_test, 1):
        result = best.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=0.25,
            iou=0.5,
            max_det=400,
            verbose=False
        )[0]

        plotted = result.plot()
        from PIL import Image as PILImage
        import cv2

        rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
        PILImage.fromarray(rgb).save(
            test_demo_dir / f"{img_path.stem}_pred.jpg",
            quality=90
        )

        if i == 1 or i % 5 == 0 or i == len(selected_test):
            log(f"[9/10] qualitative test {i}/{len(selected_test)}")

log("[9/10] Full validation + demos complete")

[13:41:10] [9/10] Full validation + count demo started
Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.3 ms, read: 789.3±612.0 MB/s, size: 645.6 KB)
val: Scanning /kaggle/working/crowddensity_full_v4/crowdhuman_yolo/labels/val.cache... 4370 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4370/4370 1.4Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 274/274 4.8it/s 56.8s
                   all       4370      99481      0.837      0.697      0.809      0.489
Speed: 0.6ms preprocess, 3.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
{
  "precision": 0.8370084562118518,
  "recall": 0.6967773970140276,
  "mAP50": 0.8092446719459494,
  "mAP50_95": 0.48899423011949256
}
[13:42:13] [9/10] counting demo 1/50 | G

In [13]:
log("[10/10] Packaging FINAL FULL-DATA artifacts")

FINAL_MODEL = OUTPUT_DIR / "crowddensity_yolo11n_full_best.pt"

shutil.copy2(
    BEST_MODEL,
    FINAL_MODEL
)

# Copy compact reports into final output directory.
FINAL_REPORT_DIR = OUTPUT_DIR / "reports"
FINAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

for report_file in REPORT_DIR.glob("*"):
    if report_file.is_file():
        shutil.copy2(
            report_file,
            FINAL_REPORT_DIR / report_file.name
        )

# Copy useful Ultralytics plots without copying the temporary dataset.
RUN_DIR = (
    WORKDIR
    / "runs"
    / "crowdhuman_yolo11n_full_v4"
)

PLOT_DIR = OUTPUT_DIR / "training_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

for name in [
    "results.png",
    "results.csv",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
    "labels.jpg",
]:
    src = RUN_DIR / name
    if src.exists():
        shutil.copy2(
            src,
            PLOT_DIR / src.name
        )

summary = {
    "project": "7% — CrowdDensity Mini",
    "version": "CrowdHuman FULL DATA YOLO11n v4",
    "task": "crowded person detection + counting baseline",
    "model": "YOLO11n",
    "train_images_used": int(len(train_manifest)),
    "val_images_used": int(len(val_manifest)),
    "all_annotated_train_used": bool(len(train_manifest) == 15000),
    "all_annotated_val_used": bool(len(val_manifest) == 4370),
    "epochs": EPOCHS,
    "img_size": IMG_SIZE,
    "batch": BATCH,
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "mAP50": metrics["mAP50"],
    "mAP50_95": metrics["mAP50_95"],
    "count_demo_images": int(len(count_df)),
    "demo_count_mae": count_mae,
    "test_folder_note": (
        "Official CrowdHuman test images were used only for qualitative "
        "inference because annotation_test.odgt is not present."
    ),
    "limitation": (
        "Person detection/counting baseline, not a density-map estimator. "
        "Severe occlusion, tiny persons, threshold choice, and domain shift "
        "may reduce performance."
    )
}

(OUTPUT_DIR / "final_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print(json.dumps(summary, indent=2), flush=True)

zip_path = shutil.make_archive(
    "/kaggle/working/crowddensity_7pct_FULL_DATA_v4_outputs",
    "zip",
    root_dir=OUTPUT_DIR
)

log(f"[10/10] Model={FINAL_MODEL}")
log(f"[10/10] ZIP={zip_path}")
log("[DONE] FULL CROWDHUMAN ANNOTATED DATA PIPELINE FINISHED")

[13:42:35] [10/10] Packaging FINAL FULL-DATA artifacts
{
  "project": "7% \u2014 CrowdDensity Mini",
  "version": "CrowdHuman FULL DATA YOLO11n v4",
  "task": "crowded person detection + counting baseline",
  "model": "YOLO11n",
  "train_images_used": 15000,
  "val_images_used": 4370,
  "all_annotated_train_used": true,
  "all_annotated_val_used": true,
  "epochs": 15,
  "img_size": 640,
  "batch": 16,
  "precision": 0.8370084562118518,
  "recall": 0.6967773970140276,
  "mAP50": 0.8092446719459494,
  "mAP50_95": 0.48899423011949256,
  "count_demo_images": 50,
  "demo_count_mae": 5.34,
  "test_folder_note": "Official CrowdHuman test images were used only for qualitative inference because annotation_test.odgt is not present.",
  "limitation": "Person detection/counting baseline, not a density-map estimator. Severe occlusion, tiny persons, threshold choice, and domain shift may reduce performance."
}
[13:42:36] [10/10] Model=/kaggle/working/crowddensity_full_v4_output/crowddensity_yolo11n